# 10 · Bias checks and the final Serengeti model

09's best model (Serengeti only, regularized XGBoost, AlphaEarth embeddings + 6 habitat covariates) reached region AUC 0.62 and camera rho 0.49 with the camera region held out. Two doubts were left open:

1. **Road bias.** GBIF sightings come from people in vehicles, so the model may still partly learn "close to a road". This is tested with distance to the nearest road as a **bias covariate**.
2. **Elevation as a stand-in for location.** Elevation and slope were the covariates the model used most. In the Serengeti, elevation changes steadily across the park, so it could be acting as a "where am I" signal rather than habitat.

Then the final model is chosen with the same rule as before, region AUC, and mapped.

Data from `buffalo_data.py`, checks from `buffalo_validation.py`. The Serengeti data, covariates and map grid are reused from 09's cache.

## 0. Data

09's Serengeti dataset and cameras, plus one new column: `dist_road_m`, the mean distance to the nearest road inside the 700 m buffer.

The roads come from **GRIP4** (Global Roads Inventory Project) through Earth Engine. GRIP maps the **public road network**: the main gravel roads through the park, such as Naabi Hill–Seronera–Lobo. It does **not** include the many game-drive tracks. So this tests bias toward main roads; bias toward small tracks remains untested.

In [1]:
import ee
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

import buffalo_data as bd
import buffalo_validation as bv
from buffalo_data import BAND_COLS, COVARIATE_COLS, ROAD_COL, BuffaloConfig, build_buffalo_dataset

EE_PROJECT = "project-0cf410a3-f35f-4911-9c5"
ee.Initialize(project=EE_PROJECT)

config = BuffaloConfig(cache_dir="cache")
data = build_buffalo_dataset(config)
areas = bv.fetch_protected_areas()

points = bv.add_utm(data.sdm_df, config.cell_size_m)
points["area"] = bv.protected_area_of(points, areas)
serengeti, ratio = bv.year_matched(points[points["area"].isin(bv.SERENGETI_AREAS)])
serengeti = bd.cached(config, "serengeti_covariates", lambda: bd.add_covariates(serengeti, "point_id", config))
serengeti = bd.cached(config, "serengeti_roads", lambda: bd.add_road_distance(serengeti, "point_id", config))

cam = bv.add_utm(data.cameras, config.cell_size_m)
cam = bd.cached(config, "camera_covariates", lambda: bd.add_covariates(cam, "SiteID", config))
cam = bd.cached(config, "camera_roads", lambda: bd.add_road_distance(cam, "SiteID", config))

print(f"Serengeti: {int(serengeti['presence'].sum())} presences, {int((serengeti['presence'] == 0).sum())} background; {len(cam)} cameras")
print("missing road distance:", serengeti[ROAD_COL].isna().sum(), "points,", cam[ROAD_COL].isna().sum(), "cameras")

presences: loaded from cache\buffalo_presences_8be5ffc111.parquet
buffalo sightings in box: 674
presences after thinning: 658
mammals: loaded from cache\buffalo_mammals_8be5ffc111.parquet
mammal sightings in box: 15873
visited cells: 334


points_embedded: loaded from cache\buffalo_points_embedded_8be5ffc111.parquet
points missing embeddings: 0
cameras_embedded: loaded from cache\buffalo_cameras_embedded_8be5ffc111.parquet
dataset: 658 presences, 6580 background, 222 cameras


serengeti_covariates: loaded from cache\buffalo_serengeti_covariates_8be5ffc111.parquet
serengeti_roads: loaded from cache\buffalo_serengeti_roads_8be5ffc111.parquet
camera_covariates: loaded from cache\buffalo_camera_covariates_8be5ffc111.parquet
camera_roads: loaded from cache\buffalo_camera_roads_8be5ffc111.parquet
Serengeti: 241 presences, 2410 background; 222 cameras
missing road distance: 0 points, 0 cameras


## 1. Are the sightings biased toward roads?

**First look:** how far from a road are the sightings, the background and the cameras? The camera traps sit on a systematic grid, but it was laid out in the central Serengeti, a well-roaded area. The table shows whether the cameras are really any further from roads than the sightings.

**Road distance by itself:** the AUC of "closer to a road = more likely a sighting", with no habitat information at all.
- **About 0.5:** sightings are no closer to roads than random places.
- **Well above 0.5:** strong road bias.

In [2]:
road_summary = pd.DataFrame({
    "presences": serengeti.loc[serengeti["presence"] == 1, ROAD_COL].describe(percentiles=[0.25, 0.5, 0.75]),
    "background": serengeti.loc[serengeti["presence"] == 0, ROAD_COL].describe(percentiles=[0.25, 0.5, 0.75]),
    "camera traps": cam[ROAD_COL].describe(percentiles=[0.25, 0.5, 0.75]),
}).loc[["25%", "50%", "75%"]]
print("distance to nearest GRIP road (m):")
print(road_summary.round(0))

road_auc = roc_auc_score(serengeti["presence"], -serengeti[ROAD_COL])
print(f"\nAUC of road distance alone (closer = sighting): {road_auc:.3f}")

distance to nearest GRIP road (m):
     presences  background  camera traps
25%     1502.0      4330.0        1652.0
50%     3547.0     10630.0        4438.0
75%     7692.0     18228.0        9871.0

AUC of road distance alone (closer = sighting): 0.715


### Correcting for it: road distance as a bias covariate

A standard correction for observer bias in presence-only models: put the bias variable **into** the model, so it explains the part of the pattern caused by access, then **fix it to one constant value** (its training median) whenever the model predicts. The map then shows the habitat part only, as if every place were equally far from a road.

Three versions, all with 09's feature set and setting:
- **habitat (09):** embeddings + covariates, no road information. This row should reproduce 09.
- **habitat + road:** road distance as an ordinary feature, a model that *uses* the bias.
- **habitat + road, fixed when predicting:** the correction.

**Who referees?** The cameras are effort-corrected, but the table above shows they are nearly as close to roads as the sightings. So they can't fully settle road bias: a road-related signal could be real buffalo behaviour, a side effect of roads following valleys and rivers, or bias that the cameras share. Read the camera rho here as "does the correction break agreement with independent data", not as proof either way.

In [3]:
HABITAT = BAND_COLS + COVARIATE_COLS
SETTING = bv.XGB_SETTINGS["regularized"]
K_REGIONS = 4

VARIANTS = {
    "habitat (09)": dict(features=HABITAT),
    "habitat + road": dict(features=HABITAT + [ROAD_COL]),
    "habitat + road, fixed when predicting": dict(features=HABITAT + [ROAD_COL], fixed_at_prediction=[ROAD_COL]),
}

road_results = {name: bv.region_evaluation(serengeti, cam, K_REGIONS, SETTING, **kw)[0] for name, kw in VARIANTS.items()}
road_comparison = pd.DataFrame(road_results).T.drop(columns=["camera p"]).astype(float).round(3)
road_comparison

,block AUC,region AUC,region TSS,calibration slope,camera rho,median rate low,median rate medium,median rate high
habitat (09),0.755,0.623,0.192,0.211,0.491,0.706,3.677,13.822
habitat + road,0.789,0.620,0.187,0.319,0.445,0.871,5.806,10.168
"habitat + road, fixed when predicting",0.714,0.596,0.160,0.109,0.404,1.062,2.748,14.934


## 2. Is elevation standing in for location?

If elevation and slope mainly tell the model *where* a point is, they help within known regions but shouldn't help in unseen ones. Four versions:
- **habitat (09):** all 6 covariates.
- **without elevation**
- **without elevation and slope**
- **location only:** just the UTM x/y coordinates, no habitat at all. This is the reference for "pure where-am-I": it should do well in 10 km block CV (it memorises hotspots) and poorly in unseen regions.

If dropping elevation leaves region AUC and camera rho about the same, elevation wasn't carrying real transferable signal. If they drop clearly, it was.

In [4]:
no_elevation = [c for c in COVARIATE_COLS if c != "elevation_m"]
no_terrain = [c for c in COVARIATE_COLS if c not in ("elevation_m", "slope_deg")]

LOCATION_VARIANTS = {
    "habitat (09)": BAND_COLS + COVARIATE_COLS,
    "without elevation": BAND_COLS + no_elevation,
    "without elevation and slope": BAND_COLS + no_terrain,
    "location only (x, y)": ["x", "y"],
}

location_results = {name: bv.region_evaluation(serengeti, cam, K_REGIONS, SETTING, features)[0]
                    for name, features in LOCATION_VARIANTS.items()}
location_comparison = pd.DataFrame(location_results).T.drop(columns=["camera p"]).astype(float).round(3)
location_comparison

,block AUC,region AUC,region TSS,calibration slope,camera rho,median rate low,median rate medium,median rate high
habitat (09),0.755,0.623,0.192,0.211,0.491,0.706,3.677,13.822
without elevation,0.746,0.626,0.198,0.193,0.408,0.932,4.423,9.764
without elevation and slope,0.721,0.595,0.106,0.085,0.342,2.031,1.776,11.451
"location only (x, y)",0.833,0.664,0.176,0.553,0.360,1.397,5.520,12.633


## 3. Final model and map

The final model is chosen with the same rule as in 08 and 09: the highest **region AUC** (unseen regions) among the habitat models.

Two versions can't be chosen:
- "habitat + road" (unfixed), because its map would partly show roads.
- "location only", because it isn't a habitat model.

The camera rho stays an independent check, never used to choose.

**Ties:** a region-AUC difference under about 0.01 is within noise for four regions. If the top candidates are that close, the choice barely matters. The cell after the map measures how similar their maps are.

The map uses 09's cached grid (Serengeti protected areas, 2.5 km, 2025) plus road distance where the chosen model needs it. Instead of thousands of grid points, the map shows **3 compact hotspots**:
1. **Smooth:** each grid point's percentile is averaged with its neighbours over about `SMOOTH_SIGMA_M` (3 km, in meters). One high-scoring point among low ones doesn't make an area.
2. **Separate peaks:** take the best remaining point; its area is every top-`HOT_PERCENT` (10 %) point within `PEAK_RADIUS_M` (4 km) of it. Remove those and repeat `N_AREAS` (3) times. Areas can't merge into one large blob this way.
3. **Draw:** each area from its 2.5 km grid squares, numbered 1 (darkest) to 3.

**Camera traps are context, not part of the ranking.** Notebook 12 tested mixing them into the score, and it made the ranking worse at cameras the model hadn't seen. So they're shown as dots sized by buffalo photo rate, and each area's tooltip lists the cameras inside it. Both layers can be switched on and off.

Read the ranking as "more likely vs less likely", not a probability. The areas only cover the visited Serengeti, where the model was tested.

In [5]:
CANDIDATES = {
    "habitat (09)": dict(features=HABITAT),
    "habitat + road, fixed when predicting": dict(features=HABITAT + [ROAD_COL], fixed_at_prediction=[ROAD_COL]),
    "without elevation": dict(features=BAND_COLS + no_elevation),
    "without elevation and slope": dict(features=BAND_COLS + no_terrain),
}
candidate_table = pd.concat([road_comparison, location_comparison]).loc[list(CANDIDATES)]
candidate_table = candidate_table[~candidate_table.index.duplicated()]
final_name = candidate_table["region AUC"].idxmax()
final = CANDIDATES[final_name]
print("final model:", final_name)
candidate_table[["block AUC", "region AUC", "region TSS", "calibration slope", "camera rho"]]

final model: without elevation


,block AUC,region AUC,region TSS,calibration slope,camera rho
habitat (09),0.755,0.623,0.192,0.211,0.491
"habitat + road, fixed when predicting",0.714,0.596,0.160,0.109,0.404
without elevation,0.746,0.626,0.198,0.193,0.408
without elevation and slope,0.721,0.595,0.106,0.085,0.342


In [6]:
import folium

MAP_YEAR = 2025
GRID_SPACING_M = 2_500
SMOOTH_SIGMA_M = 3_000
HOT_PERCENT = 10
PEAK_RADIUS_M = 4_000
N_AREAS = 3

grid = bd.cached(config, f"serengeti_grid_{MAP_YEAR}_{GRID_SPACING_M}",
                 lambda: bv.serengeti_map_grid(data.visited_cells, areas, config, MAP_YEAR, GRID_SPACING_M))
grid = bd.cached(config, f"serengeti_grid_roads_{MAP_YEAR}_{GRID_SPACING_M}", lambda: bd.add_road_distance(grid, "point_id", config))
grid = grid.dropna(subset=final["features"]).reset_index(drop=True)

model = bv.make_model(serengeti["presence"], SETTING).fit(serengeti[final["features"]], serengeti["presence"])
if final.get("fixed_at_prediction"):
    model = bv.FixedAtPrediction(model, {c: serengeti[c].median() for c in final["fixed_at_prediction"]})
grid["score"] = model.predict_proba(grid[final["features"]])[:, 1]
areas_table = bv.hotspot_areas(grid, grid["score"], cam, GRID_SPACING_M, SMOOTH_SIGMA_M, HOT_PERCENT, PEAK_RADIUS_M, N_AREAS)

print(f"final model: {final_name}; {len(areas_table)} areas")
print(areas_table.drop(columns="geometry").round(3).to_string(index=False))
print(f"\nfor comparison, median camera rate over all cameras: {cam['rate'].median():.2f} photos per 100 trap-days")

light, dark = np.array([0x86, 0xb6, 0xef]), np.array([0x0d, 0x36, 0x6b])
colour = lambda rank: "#%02x%02x%02x" % tuple((dark + (light - dark) * (rank - 1) / max(len(areas_table) - 1, 1)).astype(int))

area_map = folium.Map(location=[grid["decimalLatitude"].mean(), grid["decimalLongitude"].mean()], zoom_start=8,
                      tiles="OpenStreetMap")
area_layer = folium.FeatureGroup(name="best buffalo areas (model)").add_to(area_map)
for _, a in areas_table.iterrows():
    c = colour(a["rank"])
    camera_note = (f"{a['cameras']} cameras inside, median {a['camera median rate']:.1f} buffalo photos per 100 trap-days"
                   if a["cameras"] else "no cameras inside")
    folium.GeoJson(a.geometry.__geo_interface__,
                   style_function=lambda _, c=c: {"fillColor": c, "color": c, "weight": 1.5, "fillOpacity": 0.55},
                   tooltip=f"area {a['rank']}: {a['area_km2']:.0f} km², average percentile {a['percentile']:.0f}; {camera_note}").add_to(area_layer)
    folium.Marker([a["peak_lat"], a["peak_lon"]], icon=folium.DivIcon(
        html=f'<div style="font:bold 13px sans-serif;color:#fff;background:{c};border-radius:50%;width:22px;height:22px;'
             f'line-height:22px;text-align:center;border:1.5px solid #fff">{a["rank"]}</div>',
        icon_size=(22, 22), icon_anchor=(11, 11))).add_to(area_layer)

camera_layer = folium.FeatureGroup(name="camera traps, sized by buffalo photo rate (2010-2013)").add_to(area_map)
rate_cap = cam["rate"].quantile(0.95)
for _, c in cam.iterrows():
    folium.CircleMarker([c["decimalLatitude"], c["decimalLongitude"]], radius=2 + 6 * min(c["rate"], rate_cap) / rate_cap,
                        color="#1f1f1f", weight=1, fill=True, fill_color="#1f1f1f", fill_opacity=0.6,
                        tooltip=f"camera {c['SiteID']}: {c['rate']:.1f} buffalo photos per 100 trap-days").add_to(camera_layer)
folium.LayerControl(collapsed=False).add_to(area_map)
area_map


serengeti_grid_2025_2500: loaded from cache\buffalo_serengeti_grid_2025_2500_8be5ffc111.parquet
serengeti_grid_roads_2025_2500: loaded from cache\buffalo_serengeti_grid_roads_2025_2500_8be5ffc111.parquet


final model: without elevation; 3 areas
 rank  area_km2  percentile  peak_lon  peak_lat  cameras  camera median rate
    1     56.25      97.912    34.855    -2.498       12               8.979
    2     43.75      96.308    34.877    -2.543        7               0.000
    3     43.75      97.555    34.877    -2.453        7              15.749

for comparison, median camera rate over all cameras: 4.55 photos per 100 trap-days


In [7]:
from scipy.stats import spearmanr

baseline = bv.make_model(serengeti["presence"], SETTING).fit(serengeti[HABITAT], serengeti["presence"])
baseline_score = baseline.predict_proba(grid[HABITAT])[:, 1]
top_final = grid["score"] >= grid["score"].median()
top_baseline = baseline_score >= np.median(baseline_score)

print(f"final model ({final_name}) vs 09's model, on the map grid:")
print(f"  rank agreement: Spearman rho {spearmanr(grid['score'], baseline_score)[0]:.3f}")
print(f"  overlap of their top-50 % areas: {(top_final & top_baseline).sum() / top_final.sum():.0%}")

final model (without elevation) vs 09's model, on the map grid:
  rank agreement: Spearman rho 0.955
  overlap of their top-50 % areas: 92%
